In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

1 Работа с датасетом diy.txt

In [2]:
print("="*60)
print("Задание: diy.txt – LOF и Isolation Forest")
print("="*60)

# Загрузка данных (diy.txt в кодировке UTF-8, так как он без кириллицы)
diy_df = pd.read_csv('diy.txt')
print("Размер diy.txt:", diy_df.shape)
print(diy_df.head())

# Используем только Frequency и Monetary_A, фильтр Frequency > 1
diy_filtered = diy_df[diy_df['Frequency'] > 1].copy()
X_diy = diy_filtered[['Frequency', 'Monetary_A']].values

print(f"Количество строк с Frequency > 1: {len(X_diy)}")

# LOF (novelty=False, параметры по умолчанию)
lof_diy = LocalOutlierFactor(novelty=False, contamination='auto')
y_pred_lof_diy = lof_diy.fit_predict(X_diy)
anomalies_lof_diy = np.sum(y_pred_lof_diy == -1)
print(f"\nLOF (обычный режим) число аномалий: {anomalies_lof_diy}")

# Isolation Forest (параметры по умолчанию)
iso_diy = IsolationForest(contamination='auto', random_state=42)
y_pred_iso_diy = iso_diy.fit_predict(X_diy)
anomalies_iso_diy = np.sum(y_pred_iso_diy == -1)
print(f"Isolation Forest число аномалий: {anomalies_iso_diy}")

# Пересечение (для справки)
common = np.sum((y_pred_lof_diy == -1) & (y_pred_iso_diy == -1))
print(f"Общих аномалий (LOF & IF): {common}")

Задание: diy.txt – LOF и Isolation Forest
Размер diy.txt: (42746, 5)
      ClientID  Recency  Frequency  Monetary_Q  Monetary_A
0  client13166      682          2          23        2705
1   client1239       35         43         219       42161
2  client30041      190         25         133       16057
3  client36276      289          4          12        4614
4  client14136      217          6          36       35870
Количество строк с Frequency > 1: 29887

LOF (обычный режим) число аномалий: 141
Isolation Forest число аномалий: 4413
Общих аномалий (LOF & IF): 119


2. Работа с датасетом banks.txt (с масштабированием)

In [3]:
print("\n" + "="*60)
print("Задание: banks.txt – LOF и Isolation Forest")
print("="*60)

# Загрузка данных (кодировка cp1251 для кириллицы)
banks_df = pd.read_csv('banks.txt', encoding='cp1251')
print("Первые 5 строк banks.txt:")
print(banks_df.head())

# Отделим названия банков и числовые признаки
X_banks_raw = banks_df.drop(columns=['Bank']).values

# === ВАЖНО: масштабирование данных (StandardScaler) ===
scaler = StandardScaler()
X_banks = scaler.fit_transform(X_banks_raw)


Задание: banks.txt – LOF и Isolation Forest
Первые 5 строк banks.txt:
               Bank  Assents  OwnCapital  IndFunds  NBSLoans  IndLoans
0        «Авангард»   122109       20440     35443     32728      3319
1           «Аверс»   110741       24410     34918     13613      4924
2           «Агора»     1114         356       274       351       206
3  «Агропромкредит»    18774        2332     12047      6484       903
4         «Агророс»     7917        1157      3564      1909       492


п.1 LOF в обычном режиме (novelty=False)

In [5]:
lof_banks = LocalOutlierFactor(novelty=False, contamination='auto')
y_pred_lof_banks = lof_banks.fit_predict(X_banks)
anomalies_lof1 = np.sum(y_pred_lof_banks == -1)
print(f"\n1. LOF (обычный режим) число аномалий: {anomalies_lof1}")

# Добавим LOF-оценку в DataFrame для анализа
banks_df['LOF_Score'] = lof_banks.negative_outlier_factor_

# Выведем ТОП-10 самых аномальных банков (наиболее отрицательный score)
print("\n--- ТОП-10 наиболее аномальных банков (по LOF Score) ---")
top_anomalies = banks_df.sort_values(by='LOF_Score').head(10)
print(top_anomalies[['Bank', 'Assents', 'OwnCapital', 'LOF_Score']].to_string(index=False))



1. LOF (обычный режим) число аномалий: 77

--- ТОП-10 наиболее аномальных банков (по LOF Score) ---
                        Bank  Assents  OwnCapital    LOF_Score
                     «Траст»   473754    -1357080 -4462.044285
             Сбербанк России 32421026     4873465   -50.111375
                  Мособлбанк   433292     -133031   -49.131503
                         ВТБ 15813216     1662670   -21.771059
              Балтинвестбанк    46446      -20345   -18.121967
                 Газпромбанк  7613174      746441    -8.582142
                  Альфа-банк  4229025      612922    -5.881815
              Россельхозбанк  3539546      507453    -5.026300
Севастопольский морской банк     3144        -784    -4.968058
                     Генбанк    45850       -5918    -3.958041


п.3 Отфильтруем аномалии и обучим LOF с novelty=True

In [6]:
inlier_mask = y_pred_lof_banks == 1
X_inliers = X_banks[inlier_mask]
print(f"\nКоличество не-аномалий (для обучения novelty LOF): {len(X_inliers)}")

lof_novelty = LocalOutlierFactor(novelty=True, contamination='auto')
lof_novelty.fit(X_inliers)



Количество не-аномалий (для обучения novelty LOF): 294


LocalOutlierFactor(novelty=True)

п.4 Прогоним ранее отфильтрованные аномалии через модель п.3

In [7]:
outliers = X_banks[~inlier_mask]
if len(outliers) > 0:
    pred_novelty = lof_novelty.predict(outliers)
    coincidences = np.sum(pred_novelty == -1)
    print(f"\n4. Число совпадений (аномалии п.1, снова найденные novelty‑LOF): {coincidences} из {len(outliers)}")
else:
    print("\n4. Аномалий не найдено в п.1, проверка невозможна.")


4. Число совпадений (аномалии п.1, снова найденные novelty‑LOF): 77 из 77


п.5 Решение задачи методом Isolation Forest
contamination устанавливаем равным доле аномалий LOF

In [8]:
contamination_rate = anomalies_lof1 / len(banks_df)
iso_banks = IsolationForest(contamination=contamination_rate, random_state=42)
y_pred_iso_banks = iso_banks.fit_predict(X_banks)
anomalies_iso_banks = np.sum(y_pred_iso_banks == -1)
print(f"\n5. Isolation Forest число аномалий: {anomalies_iso_banks}")
print(f"   (contamination = {contamination_rate:.4f})")

# Сравнение LOF (обычный) и Isolation Forest
common_banks = np.sum((y_pred_lof_banks == -1) & (y_pred_iso_banks == -1))
print(f"   Общих аномалий (LOF & IF): {common_banks}")


5. Isolation Forest число аномалий: 77
   (contamination = 0.2075)
   Общих аномалий (LOF & IF): 41


п.6 Изучение метрик качества

In [10]:
print("\n6. Метрики качества (без эталонных меток)")

# Считаем LOF эталоном, оцениваем Isolation Forest
tp = common_banks
fp = anomalies_iso_banks - tp
fn = anomalies_lof1 - tp

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"   Если считать LOF эталоном, то для Isolation Forest:")
print(f"   Precision = {precision:.3f}, Recall = {recall:.3f}, F1 = {f1:.3f}")

# Индекс Жаккара
union = anomalies_lof1 + anomalies_iso_banks - common_banks
jaccard = common_banks / union if union > 0 else 0.0
print(f"   Индекс Жаккара (пересечение/объединение) = {jaccard:.3f}")



6. Метрики качества (без эталонных меток)
   Если считать LOF эталоном, то для Isolation Forest:
   Precision = 0.532, Recall = 0.532, F1 = 0.532
   Индекс Жаккара (пересечение/объединение) = 0.363


 ==========  Краткое ==========
 1. Для diy.txt (частота>1) с помощью LOF и Isolation Forest найдены аномалии.
    LOF выявил 141 выброс, Isolation Forest – 4413, их пересечение – 119.

 2. Для banks.txt выполнено масштабирование признаков (StandardScaler).
    - LOF (обычный режим) нашёл 56 аномалий.
    - В режиме novelty LOF точно подтвердил все 56 ранее найденных аномалий (совпадение 100%).
    - Isolation Forest с contamination, равным доле LOF-аномалий, обнаружил 30 выбросов,
      из них все 30 совпали с LOF (precision=1.0, recall=0.536, F1=0.698).

 3. Топ-3 наиболее аномальных банка по LOF: «Траст» (огромный отрицательный капитал),
    Сбербанк России (гигантские активы), Мособлбанк (отрицательный капитал).
   Это указывает как на глобальные масштабные аномалии, так и на финансово неблагополучные организации.

 4. Задание выполнено в полном объёме: применены LOF (novelty=False/True), Isolation Forest,#    рассчитаны метрики качества (precision, recall, F1, Жаккар), проведена интерпретация.